<a href="https://colab.research.google.com/drive/1eQOkYB4XMQdSeHV_pKvt9kSUW8vQo6LQ">Abre este Jupyter en Google Colab</a>

# Regresión Logística: Detección de SPAM

En este ejercicio se muestran los fundamentos de la Regresión Logística planteando uno de los primeros problemas que fueron solucionados mediante el uso de técnicas de Machine Learning: la detección de SPAM.

## Enunciado del ejercicio

Se propone la construcción de un sistema de aprendizaje automático capaz de predecir si un correo determinado se corresponde con un correo de SPAM o no, para ello, se utilizará el siguiente conjunto de datos:

##### [2007 TREC Public Spam Corpus](https://plg.uwaterloo.ca/cgi-bin/cgiwrap/gvcormac/foo07)
The corpus trec07p contains 75,419 messages:

    25220 ham
    50199 spam

These messages constitute all the messages delivered to a particular
server between these dates:

    Sun, 8 Apr 2007 13:07:21 -0400
    Fri, 6 Jul 2007 07:04:53 -0400

### 0. Imports

In [1]:
# Instalación de librerías externas
!pip install scikit-learn
!pip install nltk
import pandas as pd


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\colmi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\colmi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### 1. Funciones complementarias

En este caso práctico relacionado con la detección de correos electrónicos de SPAM, el conjunto de datos que disponemos esta formado por correos electrónicos, con sus correspondientes cabeceras y campos adicionales. Por lo tanto, requieren un preprocesamiento previo a que sean ingeridos por el algoritmo de Machine Learning.

In [2]:
# Esta clase facilita el preprocesamiento de correos electrónicos que poseen código HTML
from html.parser import HTMLParser

class MLStripper(HTMLParser):
    def __init__(self):
        self.reset()
        self.strict = False
        self.convert_charrefs = True
        self.fed = []

    def handle_data(self, d):
        self.fed.append(d)

    def get_data(self):
        return ''.join(self.fed)

In [3]:
# Esta función se encarga de elimar los tags HTML que se encuentren en el texto del correo electrónico
def strip_tags(html):
    s = MLStripper()
    s.feed(html)
    return s.get_data()

In [4]:
# Ejemplo de eliminación de los tags HTML de un texto
t = '<tr><td align="left"><a href="../../issues/51/16.html#article">Phrack World News</a></td>'
strip_tags(t)

'Phrack World News'

Además de eliminar los posibles tags HTML que se encuentren en el correo electrónico, deben realizarse otras acciones de preprocesamiento para evitar que los mensajes contengan ruido innecesario. Entre ellas se encuentra la eliminación de los signos de puntuación, eliminación de posibles campos del correo electrónico que no son relevantes o eliminación de los afijos de una palabra manteniendo únicamente la raiz de la misma (Stemming). La clase que se muestra a continuación realiza estas transformaciones.

In [5]:
import email
import string
import nltk
nltk.download('stopwords')

class Parser:

    def __init__(self):
        self.stemmer = nltk.PorterStemmer()
        self.stopwords = set(nltk.corpus.stopwords.words('english'))
        self.punctuation = list(string.punctuation)

    def parse(self, email_path):
        """Parse an email."""
        with open(email_path, errors='ignore') as e:
            msg = email.message_from_file(e)
        return None if not msg else self.get_email_content(msg)

    def get_email_content(self, msg):
        """Extract the email content."""
        subject = self.tokenize(msg['Subject']) if msg['Subject'] else []
        body = self.get_email_body(msg.get_payload(),
                                   msg.get_content_type())
        content_type = msg.get_content_type()
        # Returning the content of the email
        return {"subject": subject,
                "body": body,
                "content_type": content_type}

    def get_email_body(self, payload, content_type):
        """Extract the body of the email."""
        body = []
        if type(payload) is str and content_type == 'text/plain':
            return self.tokenize(payload)
        elif type(payload) is str and content_type == 'text/html':
            return self.tokenize(strip_tags(payload))
        elif type(payload) is list:
            for p in payload:
                body += self.get_email_body(p.get_payload(),
                                            p.get_content_type())
        return body

    def tokenize(self, text):
        """Transform a text string in tokens. Perform two main actions,
        clean the punctuation symbols and do stemming of the text."""
        for c in self.punctuation:
            text = text.replace(c, "")
        text = text.replace("\t", " ")
        text = text.replace("\n", " ")
        tokens = list(filter(None, text.split(" ")))
        # Stemming of the tokens
        return [self.stemmer.stem(w) for w in tokens if w not in self.stopwords]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\colmi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


##### Lectura de un correo en formato raw

In [6]:
# Cargar el archivo CSV
df = pd.read_csv('processed_data.csv')
print("Dataset cargado:")
print(f"Total de ejemplos: {len(df)}")
print(f"\nPrimeras filas:")
print(df.head())

Dataset cargado:
Total de ejemplos: 75419

Primeras filas:
   label                                            subject  \
0      1                  Generic Cialis, branded quality@    
1      0                             Typo in /debian/README   
2      1                                   authentic viagra   
3      1                               Nice talking with ya   
4      1  or trembling; stomach cramps; trouble in sleep...   

                          email_to  \
0        the00@speedy.uwaterloo.ca   
1  debian-mirrors@lists.debian.org   
2         <the00@plg.uwaterloo.ca>   
3         opt4@speedy.uwaterloo.ca   
4     ktwarwic@speedy.uwaterloo.ca   

                                          email_from  \
0                 "Tomas Jacobs" <RickyAmes@aol.com>   
1         Yan Morin <yan.morin@savoirfairelinux.com>   
2  "Sheila Crenshaw" <7stocknews@tractionmarketin...   
3       "Stormy Dempsey" <vqucsmdfgvsg@ruraltek.com>   
4         "Christi T. Jernigan" <dcube@totalink.net> 

##### Parsing del correo electrónico

In [7]:
p = Parser()
# Procesar el primer mensaje como ejemplo
sample_idx = 0
subject = str(df['subject'].iloc[sample_idx]) if pd.notna(df['subject'].iloc[sample_idx]) else ""
message = str(df['message'].iloc[sample_idx]) if pd.notna(df['message'].iloc[sample_idx]) else ""
sample_text = subject + " " + message
print(f"Label: {df['label'].iloc[sample_idx]} (0=ham, 1=spam)")
print(f"Texto original (primeros 200 caracteres):\n{sample_text[:200]}")

Label: 1 (0=ham, 1=spam)
Texto original (primeros 200 caracteres):
Generic Cialis, branded quality@  Content-Type: text/html;
Content-Transfer-Encoding: 7Bit Do you feel the pressure to perform and not rising to the occasion?? Try V ia gr a ..... your anxiety will be


##### Lectura del índice

Estas funciones complementarias se encargan cargar en memoria la ruta de cada correo electrónico y su etiqueta correspondiente {spam, ham}

In [8]:
# Distribucion de clases en el dataset
print("Distribución de clases:")
print(df['label'].value_counts())
print(f"\nPorcentaje de SPAM: {df['label'].mean()*100:.1f}%")

Distribución de clases:
label
1    50199
0    25220
Name: count, dtype: int64

Porcentaje de SPAM: 66.6%


In [9]:
import os

def create_prep_dataset_from_df(dataframe, parser, limit=None):
    """Procesar dataframe y crear dataset preprocessado"""
    X = []
    y = []
    
    if limit:
        dataframe = dataframe.head(limit)
    
    for idx, row in dataframe.iterrows():
        try:
            # Combinar subject y message
            subject_text = str(row['subject']) if pd.notna(row['subject']) else ""
            message_text = str(row['message']) if pd.notna(row['message']) else ""
            combined_text = subject_text + " " + message_text
            
            # Tokenizar
            tokens = parser.tokenize(combined_text)
            processed_text = " ".join(tokens)
            
            X.append(processed_text)
            y.append(row['label'])
        except Exception as e:
            pass
    return X, y

# Procesar primeros 100 ejemplos como muestra
X_sample, y_sample = create_prep_dataset_from_df(df, p, limit=100)
print(f"Procesadas 100 muestras")

Procesadas 100 muestras


In [10]:
print(f"Ejemplos procesados: {len(X_sample)}")
print(f"Labels: {set(y_sample)}")
print(f"Distribución: Ham={y_sample.count(0)}, Spam={y_sample.count(1)}")

Ejemplos procesados: 100
Labels: {0, 1}
Distribución: Ham=23, Spam=77


In [11]:
print("Primeros 3 textos procesados:")
for i in range(min(3, len(X_sample))):
    print(f"\n{i+1}. [Label {y_sample[i]}]")
    print(f"   {X_sample[i][:150]}...")

Primeros 3 textos procesados:

1. [Label 1]
   gener ciali brand qualiti contenttyp texthtml contenttransferencod 7bit do feel pressur perform rise occas tri v ia gr anxieti thing past back old sel...

2. [Label 0]
   typo debianreadm hi ive updat gulu i check mirror it seem littl typo debianreadm file exampl httpgulususherbrookecadebianreadm ftpftpfrdebianorgdebian...

3. [Label 1]
   authent viagra contenttyp textplain charsetiso88591 contenttransferencod 7bit mega authenticv i a g r a discount pricec i a l i s discount pricedo mis...


### 2. Preprocesamiento de los datos del conjunto de datos

Con las funciones presentadas anteriormente se permite la lectura de los correos electrónicos de manera programática y el procesamiento de los mismos para eliminar aquellos componentes que no resultan de utilidad para la detección de correos de SPAM. Sin embargo, cada uno de los correos sigue estando representado por un diccionario de Python con una serie de palabras.

In [12]:
# Usar la muestra de 100 como conjunto de entrenamiento
X_train = X_sample
y_train = y_sample
print(f"Conjunto de entrenamiento: {len(X_train)} ejemplos")

Conjunto de entrenamiento: 100 ejemplos


In [13]:
print(f"Textos en X_train: {len(X_train)}")
print(f"Labels en y_train: {len(y_train)}")

Textos en X_train: 100
Labels en y_train: 100


In [14]:
# Procesar todo el dataset para secciones posteriores
print("Procesando todo el dataset...")
X_full, y_full = create_prep_dataset_from_df(df, p)
print(f"Dataset completo: {len(X_full)} ejemplos")
print(f"Distribution: Ham={y_full.count(0)}, Spam={y_full.count(1)}")

Procesando todo el dataset...
Dataset completo: 75419 ejemplos
Distribution: Ham=25220, Spam=50199


El algoritmo de Regresión Logística no es capaz de ingerir texto como parte del conjunto de datos. Por lo tanto, deben aplicarse una serie de funciones adicionales que transformen el texto de los correos electrónicos parseados en una representación numérica.

##### Aplicación de CountVectorizer

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

# Preparación del email en una cadena de texto usando datos ya procesados
if len(X_sample) == 0:
    raise ValueError("X_sample está vacío. Ejecuta primero las celdas de carga y preprocesamiento.")

prep_email = [X_sample[0]]

vectorizer = CountVectorizer()
X = vectorizer.fit(prep_email)

print("Email (primeros 200 caracteres):", prep_email[0][:200], "...\n")
print("Características de entrada (primeras 20):", vectorizer.get_feature_names_out()[:20])

Email (primeros 200 caracteres): gener ciali brand qualiti contenttyp texthtml contenttransferencod 7bit do feel pressur perform rise occas tri v ia gr anxieti thing past back old self ...

Características de entrada (primeras 20): ['7bit' 'anxieti' 'back' 'brand' 'ciali' 'contenttransferencod'
 'contenttyp' 'do' 'feel' 'gener' 'gr' 'ia' 'occas' 'old' 'past' 'perform'
 'pressur' 'qualiti' 'rise' 'self']


In [21]:
X = vectorizer.transform(prep_email)
print("\nValues:\n", X.toarray())


Values:
 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]


##### Aplicación de OneHotEncoding

In [19]:
from sklearn.preprocessing import OneHotEncoder

# Usar los tokens del primer ejemplo del dataset procesado
tokens_sample = X_sample[0].split()[:10]  # Primeros 10 tokens
print(f"Tokens de ejemplo: {tokens_sample}")

# Aplicar OneHotEncoding
prep_tokens = [[w] for w in tokens_sample]

enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_onehot = enc.fit_transform(prep_tokens)

print(f"\nFeatures (tokens únicos): {enc.get_feature_names_out()}")
print(f"\nMatriz OneHot (primeros 5 tokens):")
print(X_onehot[:5])

Tokens de ejemplo: ['gener', 'ciali', 'brand', 'qualiti', 'contenttyp', 'texthtml', 'contenttransferencod', '7bit', 'do', 'feel']

Features (tokens únicos): ['x0_7bit' 'x0_brand' 'x0_ciali' 'x0_contenttransferencod' 'x0_contenttyp'
 'x0_do' 'x0_feel' 'x0_gener' 'x0_qualiti' 'x0_texthtml']

Matriz OneHot (primeros 5 tokens):
[[0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]


##### Funciones auxiliares para preprocesamiento del conjunto de datos

In [20]:
def create_prep_dataset(index_path, n_elements):
    X = []
    y = []
    indexes = parse_index(index_path, n_elements)
    for i in range(n_elements):
        print("\rParsing email: {0}".format(i+1), end='')
        try:
            mail, label = parse_email(indexes[i])
            X.append(" ".join(mail['subject']) + " ".join(mail['body']))
            y.append(label)
        except:
            pass
    return X, y

### 3. Entrenamiento del algoritmo 

In [22]:
# Usar los primeros 100 ejemplos para entrenamiento
X_train_100 = X_full[:100]
y_train_100 = y_full[:100]
print(f"Dataset de entrenamiento (100 ejemplos): {len(X_train_100)}")
print(f"X_train primeros 5 elementos preparados")

Dataset de entrenamiento (100 ejemplos): 100
X_train primeros 5 elementos preparados


##### Aplicamos la vectorización a los datos

In [23]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train_100)
print(f"Vectorización completada: {X_train_vec.shape}")

Vectorización completada: (100, 8754)


In [24]:
print("Primeras 5 filas vectorizadas:")
print(X_train_vec[:5].toarray())
print(f"\nNúmero de features: {len(vectorizer.get_feature_names_out())}")

Primeras 5 filas vectorizadas:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Número de features: 8754


In [25]:
import pandas as pd

# Mostrar como DataFrame (solo las primeras columnas por claridad)
feature_names = vectorizer.get_feature_names_out()[:20]
df_vectors = pd.DataFrame(X_train_vec[:10, :20].toarray(), columns=feature_names)
print("Vectores de los primeros 10 ejemplos (primeras 20 features):")
df_vectors

Vectores de los primeros 10 ejemplos (primeras 20 features):


,0000,000301c634d35e87f4f0aa0fa8c0sanya,000701c6c5daae97e3900100a8c0alex,00085,000f01c7797d598c5370067082f4hannah,000f01c77a12c837f96006babfd4g7e986bc2c9d74,001301c77a138af7c7500068792cmasternak,001501c77a148a74bf3000cb9bd4xxxqewuywf9i7x,001601c77a146b611e900019ca8cdzieci,001b01c77a1b68f8a5e005604a14knuttel6vwc576,001c01c77a12c744e6d006c9c734aska,003,00450,01,0107,014,01417,020,023,02mtzi2oxls6tnelitizjlzzezyuhtdl9xzg4719wvbsbxleu17fzefbgayqijrxz5djvkkdr
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [26]:
print("Labels de entrenamiento (y_train):")
print(f"Total: {len(y_train_100)}")
print(f"Ham (0): {y_train_100.count(0)}")
print(f"Spam (1): {y_train_100.count(1)}")

Labels de entrenamiento (y_train):
Total: 100
Ham (0): 23
Spam (1): 77


###### Entrenamiento del algoritmo de regresión logística con el conjunto de datos preprocesado

In [27]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train_100)
print("Modelo entrenado exitosamente")

Modelo entrenado exitosamente


### 4. Predicción

##### Lectura de un conjunto de correos nuevos

In [28]:
# Usar ejemplos 100-150 para test
X_test_50 = X_full[100:150] if len(X_full) >= 150 else X_full[100:]
y_test_50 = y_full[100:150] if len(y_full) >= 150 else y_full[100:]
print(f"Dataset de test: {len(X_test_50)} ejemplos")

Dataset de test: 50 ejemplos


##### Preprocesamiento de los correos con el vectorizador creado anteriormente

In [29]:
X_test_vec = vectorizer.transform(X_test_50)
print(f"Test vectorizado: {X_test_vec.shape}")

Test vectorizado: (50, 8754)


##### Predicción del tipo de correo

In [30]:
y_pred = clf.predict(X_test_vec)
print(f"Predicciones realizadas: {len(y_pred)}")
print(f"Predicciones (primeras 10): {y_pred[:10]}")

Predicciones realizadas: 50
Predicciones (primeras 10): [1 1 1 1 1 1 1 1 1 1]


In [31]:
print("Predicción (primeras 10):\n", y_pred[:10])
print("\nEtiquetas reales (primeras 10):\n", y_test_50[:10])

Predicción (primeras 10):
 [1 1 1 1 1 1 1 1 1 1]

Etiquetas reales (primeras 10):
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


##### Evaluación de los resultados

In [32]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test_50, y_pred)
print(f'Accuracy: {accuracy:.3f}')

Accuracy: 0.980


### 5. Aumentando el conjunto de datos

In [33]:
# El dataset ya está procesado en X_full, y_full
# Dividir en entrenamiento (80%) y test (20%)
split_point = int(len(X_full) * 0.8)
X_train_full = X_full[:split_point]
y_train_full = y_full[:split_point]
X_test_full = X_full[split_point:]
y_test_full = y_full[split_point:]
print(f"Entrenamiento: {len(X_train_full)} ejemplos")
print(f"Test: {len(X_test_full)} ejemplos")

Entrenamiento: 60335 ejemplos
Test: 15084 ejemplos


In [34]:
# Dividir dataset completo para mejor evaluación
if len(X_full) > 10000:
    X_train_lg = X_full[:10000]
    y_train_lg = y_full[:10000]
    X_test_lg = X_full[10000:]
    y_test_lg = y_full[10000:]
else:
    # Si el dataset es más pequeño, usar 80-20 split
    split = int(len(X_full) * 0.8)
    X_train_lg = X_full[:split]
    y_train_lg = y_full[:split]
    X_test_lg = X_full[split:]
    y_test_lg = y_full[split:]

print(f"Entrenamiento: {len(X_train_lg)} ejemplos")
print(f"Test: {len(X_test_lg)} ejemplos")

Entrenamiento: 10000 ejemplos
Test: 65419 ejemplos


In [35]:
vectorizer_lg = CountVectorizer()
X_train_lg_vec = vectorizer_lg.fit_transform(X_train_lg)
print(f"Vectorización: {X_train_lg_vec.shape}")

Vectorización: (10000, 221445)


In [36]:
clf_lg = LogisticRegression(max_iter=1000)
clf_lg.fit(X_train_lg_vec, y_train_lg)
print("Modelo entrenado con dataset grande")

Modelo entrenado con dataset grande


In [37]:
X_test_lg_vec = vectorizer_lg.transform(X_test_lg)
print(f"Test vectorizado: {X_test_lg_vec.shape}")

Test vectorizado: (65419, 221445)


In [38]:
y_pred_lg = clf_lg.predict(X_test_lg_vec)
print(f"Predicciones: {len(y_pred_lg)}")

Predicciones: 65419


In [39]:
accuracy_lg = accuracy_score(y_test_lg, y_pred_lg)
print(f'Accuracy con dataset grande: {accuracy_lg:.3f}')

Accuracy con dataset grande: 0.976
